In [0]:
%python

from delta.tables import DeltaTable
from pyspark.sql import functions as F

tabela = "blueprint_spark.bronze.clientes_delta_spark"

clientes = spark.createDataFrame([
    (1, "Ana",   "Salvador", True),
    (2, "Bruno", "Recife",   True),
    (3, "Carla", "Curitiba", False),
], ["id", "nome", "cidade", "ativo"])

novos = spark.createDataFrame([
    (1, "Ana",   "Salvador", True),   # cancelou
    (2, "Bruno", "Salvador", True),    # mudou de cidade
    (4, "Diego", "Rio de Janeiro",   True),
    (5, "Walter", "Sao Paulo",   True),   # cliente novo
], ["id", "nome", "cidade", "ativo"])

In [0]:
clientes.write.mode("overwrite").saveAsTable(tabela)

In [0]:
spark.table(tabela).show()

+---+-----+--------+-----+
| id| nome|  cidade|ativo|
+---+-----+--------+-----+
|  1|  Ana|Salvador| true|
|  2|Bruno|  Recife| true|
|  3|Carla|Curitiba|false|
+---+-----+--------+-----+



In [0]:
dt = DeltaTable.forName(spark,tabela)

dt.alias("t").merge(novos.alias("n"),"t.id = n.id") \
    .whenMatchedDelete(condition="n.ativo = false") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

spark.table(tabela).show()

+---+-----+--------+-----+
| id| nome|  cidade|ativo|
+---+-----+--------+-----+
|  3|Carla|Curitiba|false|
|  2|Bruno|Salvador| true|
|  4|Diego|  Recife| true|
+---+-----+--------+-----+



In [0]:
dt = DeltaTable.forName(spark,tabela)

dt.alias("t").merge(novos.alias("n"),"t.id = n.id") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

spark.table(tabela).show()

+---+------+--------------+-----+
| id|  nome|        cidade|ativo|
+---+------+--------------+-----+
|  3| Carla|      Curitiba|false|
|  4| Diego|Rio de Janeiro| true|
|  5|Walter|     Sao Paulo| true|
|  2| Bruno|      Salvador| true|
|  1|   Ana|      Salvador| true|
+---+------+--------------+-----+



In [0]:
%python

clientes.write.mode("overwrite").saveAsTable(tabela)

dt = DeltaTable.forName(spark, tabela)

dt.update(
    condition=F.col("id") == 3,
    set={"ativo": F.lit(True)}
)

spark.table(tabela).show()
# Carla reativada

+---+-----+--------+-----+
| id| nome|  cidade|ativo|
+---+-----+--------+-----+
|  1|  Ana|Salvador| true|
|  2|Bruno|  Recife| true|
|  3|Carla|Curitiba| true|
+---+-----+--------+-----+



In [0]:
%python

clientes.write.mode("overwrite").saveAsTable(tabela)

dt = DeltaTable.forName(spark, tabela)

dt.delete(F.col("ativo") == False)

spark.table(tabela).show()
# Carla removida

+---+-----+--------+-----+
| id| nome|  cidade|ativo|
+---+-----+--------+-----+
|  1|  Ana|Salvador| true|
|  2|Bruno|  Recife| true|
+---+-----+--------+-----+



In [0]:
clientes.write.mode("overwrite").saveAsTable(tabela)

com_email = spark.createDataFrame([
    (5, "Elaine", "Recife", True, "elaine@email.com"),
], ["id", "nome", "cidade", "ativo", "email"])

com_email.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(tabela)

spark.table(tabela).show()

+---+------+------+-----+----------------+
| id|  nome|cidade|ativo|           email|
+---+------+------+-----+----------------+
|  5|Elaine|Recife| true|elaine@email.com|
+---+------+------+-----+----------------+

